# RE_106 · Quan i on es respira pitjor a Barcelona

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 3**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes a l'Agència de Salut Pública de Barcelona. Abans de la campanya sobre la zona de
baixes emissions, la direcció vol saber com es comporta la contaminació al llarg del dia i entre
barris, amb les dades horàries de totes les estacions de la ciutat durant el 2024.

**La pregunta:** a quines hores i a quines estacions l'aire és pitjor? Els caps de setmana es
respira millor? Quines estacions es comporten igual i quines no?

L'enunciat complet, amb el diccionari de columnes, és a [RE_106](../RE_106_aire_bcn.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FITXER = "aire_bcn_2024.csv"
URL = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/" + FITXER
ruta = URL
for candidata in (os.path.join("dades", FITXER), os.path.join("..", "dades", FITXER)):
    if os.path.exists(candidata):
        ruta = candidata
df = pd.read_csv(ruta)
print(ruta, df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma, estacions i contaminants

Quantes files i columnes hi ha? Quantes estacions i quants contaminants? Fes una taula amb les estacions a les files, els contaminants a les columnes i el nombre de **dies** amb dades a cada casella. Totes les estacions mesuren el mateix tot l'any?

In [ ]:
print(df.shape, df["nom_estacio"].nunique(), df["contaminant"].nunique())
dies = df.groupby(["nom_estacio", "contaminant"]).size().unstack()
print(dies.sum().sum(), len(df), dies.max().max())
dies

**Comentari.** Totes les estacions mesuren NO2 tot l'any, però no tots els contaminants: PM10 a Gràcia - Sant Gervasi només té 64 dies, i PM2.5 al Parc Vall d'Hebron, 93. Les mitjanes anuals d'aquestes estacions no són comparables amb les altres.

### 1.2 · Buits per hora

Quina proporció de buits té cada columna? Fes un gràfic de barres amb el percentatge de buits de cada hora (`h01` … `h24`). Hi ha hores on falten més dades?

In [ ]:
print(df.isna().sum()[df.isna().sum() > 0].head(3), "...")
hores = [f"h{i:02d}" for i in range(1, 25)]
pct_buits = 100 * df[hores].isna().mean()
print(pct_buits.round(1).sort_values(ascending=False).head(6))
pct_buits.plot(kind="bar", title="% de buits per hora")
plt.show()

### 1.3 · Unitats

Quina unitat té cada contaminant? Fes una taula amb el nombre de files per contaminant i unitat. Hi ha res estrany?

In [ ]:
unitats = df.groupby(["contaminant", "unitats"]).size()
print(unitats, unitats.sum())

**Comentari.** `ug/m3` i `µg/m3` són la mateixa unitat escrita de dues maneres (unes poques files de NO, NO2 i O3). El CO va en `mg/m3`: mil vegades més gran. No té sentit fer la mitjana de contaminants diferents ni posar el CO al mateix gràfic sense pensar-hi.

## Nivell 2 · Respondre

### 2.1 · Una fila per hora

Passa la taula a format llarg amb `melt`: una fila per estació, contaminant, dia i hora, amb una columna `hora` numèrica **de 0 a 23** i una columna `valor`. Converteix també `data` a data.

In [ ]:
llarg = pd.melt(df, id_vars=["nom_estacio", "contaminant", "unitats", "data"], value_vars=hores,
               var_name="h", value_name="valor")
llarg["hora"] = llarg["h"].str[1:].astype(int) - 1
llarg["data"] = pd.to_datetime(llarg["data"])
print(llarg.shape, len(df) * 24)
print(llarg["valor"].notna().sum(), df[hores].notna().sum().sum())
print(llarg["hora"].min(), llarg["hora"].max())

### 2.2 · El perfil del NO2

El diòxid de nitrogen (NO2) ve sobretot del trànsit. Calcula'n el valor mitjà per hora del dia, amb totes les estacions juntes, i dibuixa'l. A quines hores hi ha els pics?

In [ ]:
no2 = llarg[llarg["contaminant"] == "NO2"]
perfil = no2.groupby("hora")["valor"].mean()
print(len(perfil))
print(perfil.round(1).sort_values(ascending=False).head(4))
perfil.plot(marker="o", title="NO2 mitjà per hora (totes les estacions, 2024)")
plt.ylabel("µg/m³")
plt.show()

### 2.3 · L'estació pitjor

Quina estació té el NO2 mitjà més alt del 2024? I el PM10? Al costat de cada mitjana posa quants dies diferents ha mesurat l'estació aquell contaminant. Te'n refies de totes?

In [ ]:
def per_estacio(cont):
    t = llarg[llarg["contaminant"] == cont].groupby("nom_estacio").agg(
        mitjana=("valor", "mean"), dies=("data", "nunique"), primer=("data", "min"), ultim=("data", "max"))
    return t.sort_values("mitjana", ascending=False).round(1)

print(per_estacio("NO2"))
pm10 = per_estacio("PM10")
print(pm10)
# comparació justa: només els dies que Gràcia va mesurar PM10
dies_gracia = llarg.loc[(llarg["contaminant"] == "PM10") & (llarg["nom_estacio"].str.contains("Gràcia")), "data"].unique()
justa = llarg[(llarg["contaminant"] == "PM10") & llarg["data"].isin(dies_gracia)]
justa.groupby("nom_estacio")["valor"].mean().round(1).sort_values(ascending=False)

**Comentari.** La mitjana de PM10 de Gràcia - Sant Gervasi (24,4, la primera) surt de 64 dies d'hivern (de l'1-gen al 4-mar): no es pot comparar amb les estacions que han mesurat tot l'any. Comparant només aquells mateixos dies, l'Eixample passa al davant (26,7 contra 24,4).

### 2.4 · Feiners i caps de setmana

Repeteix el perfil horari del NO2 de 2.2, però separant els dies feiners (de dilluns a divendres) i els caps de setmana. Dibuixa les dues corbes al mateix gràfic. Què canvia?

In [ ]:
no2 = no2.assign(cap_setmana=no2["data"].dt.dayofweek >= 5)
fs = no2.pivot_table(index="hora", columns="cap_setmana", values="valor", aggfunc="mean")
fs.columns = ["feiner", "cap de setmana"]
print(fs.shape)
print(fs.round(1).iloc[[3, 8, 13, 20]])
fs.plot(marker=".", title="NO2 per hora: feiners i caps de setmana")
plt.ylabel("µg/m³")
plt.show()

### 2.5 · Estacions bessones (NumPy)

Cada estació té un perfil horari de NO2: 24 números, la mitjana de cada hora.

1. Fes una taula amb una fila per estació i una columna per hora, i passa-la a un array `M`.
2. Calcula la distància euclidiana entre **totes les parelles** d'estacions: una matriu `D`.
3. Quines dues estacions s'assemblen més? I quines dues són més diferents?

In [ ]:
perfils = no2.pivot_table(index="nom_estacio", columns="hora", values="valor", aggfunc="mean")
M = perfils.to_numpy()
D = np.linalg.norm(M[:, None, :] - M[None, :, :], axis=2)
print(M.shape, D.shape, np.allclose(D, D.T), np.diag(D).max())
noms = perfils.index.to_numpy()
D2 = D.copy()
np.fill_diagonal(D2, np.inf)
i, j = np.unravel_index(np.argmin(D2), D2.shape)
print("més semblants:", noms[i], "/", noms[j], round(D[i, j], 1))
i, j = np.unravel_index(np.argmax(D), D.shape)
print("més diferents:", noms[i], "/", noms[j], round(D[i, j], 1))
pd.DataFrame(D, index=noms, columns=noms).round(0)

## Nivell 3 · Més enllà

### 3.1 · Dies per sobre del límit

La normativa europea fixa un valor límit diari de PM10 de 50 µg/m³ (la mitjana del dia). Quants dies el va superar cada estació el 2024? Tingues en compte quants dies va mesurar cada una.

In [ ]:
pm = llarg[llarg["contaminant"] == "PM10"]
diaria = pm.groupby(["nom_estacio", "data"])["valor"].mean()
(diaria > 50).groupby(level=0).agg(["sum", "size"])

### 3.2 · Tots els contaminants junts

Fes el perfil horari mitjà de **tots** els contaminants en un sol gràfic. Té sentit? Com l'arreglaries perquè es puguin comparar les formes de les corbes?

In [ ]:
t = llarg.pivot_table(index="hora", columns="contaminant", values="valor", aggfunc="mean")
print(t.max().round(2))
(t / t.max()).plot(figsize=(10, 4), title="Perfil horari relatiu (1 = màxim de cada contaminant)")
plt.show()

## Conclusió de referència

El NO2, que ve del trànsit, té dos pics diaris: al matí (màxim a les 8 h, 30,0 µg/m³ de mitjana
entre totes les estacions) i al vespre (25,6 a les 20 h). En dies feiners, a les 8 h hi ha gairebé el
doble de NO2 que el cap de setmana (34,6 contra 18,6), i el pic del matí pràcticament desapareix el
cap de setmana. L'Eixample és l'estació amb més NO2 (32,5 de mitjana anual) i l'Observatori Fabra, la
més neta (7,7); són també els dos perfils horaris més diferents, mentre que Ciutadella i Poblenou són
les estacions més semblants. En PM10, Gràcia - Sant Gervasi sembla la pitjor, però només té 64 dies
d'hivern: comparant els mateixos dies, l'Eixample torna a ser la primera. Paranys: unitats escrites
de dues maneres (`ug/m3` i `µg/m3`), el CO en mg/m³, i forats que es concentren entre les 10 i les
14 h (fins a un 8,5 % a `h12`).